In [ ]:

ALPHABET : list[str] = ['A', 'C', 'G', 'T']
WORD_LENGTH : int = 8

In [ ]:
def Check_CG_Count(word : str) -> bool:
    if(word.count('C') + word.count('G') == 4):
        return True
    return False

In [ ]:
def Check_4_Position_Difference(word_x : str, word_y : str) -> bool:
    diffCount = 0
    for i in range(WORD_LENGTH):
        if(word_x[i] != word_y[i]):
            diffCount += 1

    if(diffCount >= 4):
        return True
    return False

In [ ]:
def Check_RC_Difference(word_x : str, word_y :str) -> bool:
    R_word = Get_Reverse_Word(word_x)
    C_word = Get_WCC_Word(word_y)

    if(Check_4_Position_Difference(R_word, C_word)):
        return True
    return False

def Get_Reverse_Word(word : str) -> str:
    return word[::-1]

def Get_WCC_Word(word: str) -> str:
    complements = {
        'A': 'T',
        'T': 'A',
        'C': 'G',
        'G': 'C',
    }

    return ''.join(complements[letter] for letter in word)

In [ ]:
def GenerateAllEligibleWords() -> list[str]:
    words = [""]
    for i in range(WORD_LENGTH):
        words = GenerateAllEligibleWordsHelper(words)

    return [word for word in words if(Check_CG_Count(word) and Check_RC_Difference(word, word))]

def GenerateAllEligibleWordsHelper(array : list[str]) -> list[str]:
    out = []
    for subword in array:
        for i in ALPHABET:
            out.append(subword + i)
    return out

def GenerateVariablesForWords(words : list[str]) -> dict[str : int]:
    return {
        word: index
        for index, word in enumerate(words, start = 1)
        }

In [ ]:
def GetRuleClauses(words : list[str], wordVariables : dict[str : int]) -> list[tuple[int, int]]:

    clauses = []
    for word_x in words:
        for word_y in words:
            if(word_x == word_y):
                continue
            if(not (
                Check_4_Position_Difference(word_x, word_y) and 
                Check_RC_Difference(word_x, word_y))):
                clauses.append((-wordVariables[word_x], -wordVariables[word_y]))

    return clauses

def GetSizeClauses(wordVariables : dict[str, int], targetSize : int) -> tuple[list[tuple[int, ...]], int]:
    clauses = []
    selectedVariables = list(wordVariables.values())
    wordCount = len(selectedVariables)
    nextVariable = max(selectedVariables, default = 0) + 1

    trueLiteral = nextVariable
    nextVariable += 1
    clauses.append((trueLiteral,))

    falseLiteral = -trueLiteral

    
    dp = [[falseLiteral] * (targetSize + 1) for _ in range(wordCount + 1)]

    for i in range(wordCount + 1):
        dp[i][0] = trueLiteral

    for i in range(1, wordCount + 1):
        selectedVariable = selectedVariables[i - 1]

        for j in range(1, targetSize + 1):
            currentState = nextVariable
            nextVariable += 1
            dp[i][j] = currentState

            alreadyReached = dp[i - 1][j]
            previousLevel = dp[i - 1][j - 1]

            clauses.append((-alreadyReached, currentState))
            clauses.append((-previousLevel, -selectedVariable, currentState))
            clauses.append((-currentState, alreadyReached, previousLevel))
            clauses.append((-currentState, alreadyReached, selectedVariable))

    clauses.append((dp[wordCount][targetSize],))

    return clauses, nextVariable - 1


In [ ]:
def Run(target_set_size : int):
    words = GenerateAllEligibleWords()
    wordVariables = GenerateVariablesForWords(words)
    clauses = GetRuleClauses(words, wordVariables) + GetSizeClauses(words, wordVariables)

    with open("glucoseClauses.txt", 'w') as file:
        file.write(f"p cnf {len(wordVariables)} {len(clauses)}\n")
        for clause in clauses:
            file.write(f"{clause[0]} {clause[1]} 0\n")

    
    